# 🧪 Lab 01: First Contact — Spark Discovers Shapes 🌍🚀

Welcome to Mission Control.

This is the **first spatial lab**, so we assume exactly zero GIS knowledge.

We are **not** calculating distances, reprojection, buffers, spatial indexes, or anything else that sounds expensive yet. We are asking a simpler question:

> **What is spatial data, and what exactly changes when Spark 4.2 stops seeing a shape as anonymous bytes and starts seeing it as `GEOMETRY`?**

We will meet three basic shapes:

```text
POINT       → one location
LINESTRING  → a connected path
POLYGON     → an enclosed area
```

Then we will put the **same shape bytes** through two Spark contracts:

```text
BINARY
   vs
GEOMETRY(4326)
```

Finally, we will ask Spark one real spatial question:

> **Is the Madrid store inside the delivery zone?**

That last question is where things get interesting.

### 🎯 Mission objectives

By the end of the notebook we will have proved, from the running Spark 4.2 engine, that:

- a `POINT`, `LINESTRING`, and `POLYGON` are different kinds of spatial shapes;
- WKB is a binary representation that Spark can carry even when it knows nothing about the spatial meaning;
- `BINARY` and native `GEOMETRY(4326)` can contain the same shape while exposing different type contracts;
- the native Geometry carries an SRID that Spark can inspect;
- PySpark returns a real Python `Geometry` object for the native column;
- stock Spark 4.2 understands the **spatial noun** `GEOMETRY`, but it still does not ship the **spatial verb** `ST_Within`.

No Sedona is installed in this notebook.

That is deliberate.

We want to know what **stock Spark 4.2 itself** understands before anybody adds a geospatial expansion pack.

## 0 — Pre-flight checks 🛰️

This notebook is designed for:

```text
PySpark  4.2.0
Spark    4.2.0
Java     17+
```

It does **not** need Sedona, Shapely, GeoPandas, or any other GIS library.

A couple of local-environment annoyances are handled here on purpose:

- `PYSPARK_SUBMIT_ARGS` is given the normal `pyspark-shell` default if your machine has not defined it. This avoids the Windows/local PySpark bootstrap failure where Spark dies before the JVM even starts.
- PySpark 4.2 can emit a noisy pandas 3.x compatibility warning during import even though **this notebook does not use pandas at all**. We suppress that one import-time warning and report the installed pandas version ourselves in the runtime fingerprint. Later pandas/Arrow labs should use a PySpark-compatible pandas version.

The important contract for this lab is simple:

> **Stock Spark 4.2.0, geospatial support enabled, no spatial extension loaded.**

In [1]:
import sys, json, warnings,  importlib.metadata as metadata


# This lab never uses pandas. Suppress only PySpark's known import-time
# pandas >= 3 warning so the first output is useful Mission Control data,
# not unrelated stderr noise.
warnings.filterwarnings(
    "ignore",
    message=r"PySpark does not yet fully support pandas >= 3\.0\.0.*",
    category=FutureWarning,
)

import pyspark
from pyspark.sql import SparkSession, functions as F

def installed_version(package_name):
    try:
        return metadata.version(package_name)
    except metadata.PackageNotFoundError:
        return None

active = SparkSession.getActiveSession()
if active is not None:
    active.stop()

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("lab-01-first-contact-spatial-data")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

java_version = spark.sparkContext._jvm.java.lang.System.getProperty("java.version")
pandas_version = installed_version("pandas")

fingerprint = {
    "python": sys.version.split()[0],
    "pyspark": pyspark.__version__,
    "spark": spark.version,
    "java": java_version,
    "geospatial_enabled": spark.conf.get("spark.sql.geospatial.enabled"),
    "pandas": pandas_version or "not installed",
}

print("🚀 Runtime fingerprint")
print(json.dumps(fingerprint, indent=2))

assert fingerprint["pyspark"] == "4.2.0", fingerprint
assert fingerprint["spark"] == "4.2.0", fingerprint
assert int(java_version.split(".")[0]) >= 17, fingerprint
assert fingerprint["geospatial_enabled"].lower() == "true", fingerprint

if pandas_version and pandas_version.split(".")[0].isdigit() and int(pandas_version.split(".")[0]) >= 3:
    print(
        "\n🐼 pandas note: pandas >= 3 is installed, but this lab never uses pandas. "
        "Keep it in mind for later pandas/Arrow experiments."
    )

print("\n✅ Mission Control is green. Stock Spark 4.2 spatial support is enabled.")

🚀 Runtime fingerprint
{
  "python": "3.14.0",
  "pyspark": "4.2.0",
  "spark": "4.2.0",
  "java": "17.0.19",
  "geospatial_enabled": "true",
  "pandas": "3.0.5"
}

🐼 pandas note: pandas >= 3 is installed, but this lab never uses pandas. Keep it in mind for later pandas/Arrow experiments.

✅ Mission Control is green. Stock Spark 4.2 spatial support is enabled.


# 1 — Meet the Spatial Zoo 🐘🌍

Before touching Spark internals, let's make the word **geometry** concrete.

We will use three objects around Madrid:

```text
Madrid Store
    POINT
    one location

Tiny Delivery Route
    LINESTRING
    several connected locations

Tiny Delivery Zone
    POLYGON
    one enclosed area
```

For humans, we keep a readable representation beside each object:

```text
POINT(-3.7038 40.4168)
```

That notation is called **WKT — Well-Known Text**.

Spark 4.2's tiny native constructor surface is WKB-centric, so for the machine payload we will encode the same shapes as **WKB — Well-Known Binary**.

Do not worry about the binary format yet.

Lab 02 will perform the alien autopsy.

For now:

```text
WKT → nice for our eyes
WKB → standardized bytes for the machine
```

In [2]:
import struct

def wkb_point(x, y):
    """Little-endian OGC WKB Point."""
    return struct.pack("<BIdd", 1, 1, float(x), float(y))

def wkb_linestring(points):
    """Little-endian OGC WKB LineString."""
    payload = struct.pack("<BII", 1, 2, len(points))
    payload += b"".join(struct.pack("<dd", float(x), float(y)) for x, y in points)
    return payload

def wkb_polygon(ring):
    """Little-endian OGC WKB Polygon with one exterior ring."""
    if ring[0] != ring[-1]:
        raise ValueError("Polygon ring must be closed: first point must equal last point.")
    payload = struct.pack("<BII", 1, 3, 1)          # endian, type=Polygon, ring count
    payload += struct.pack("<I", len(ring))
    payload += b"".join(struct.pack("<dd", float(x), float(y)) for x, y in ring)
    return payload

madrid = (-3.7038, 40.4168)

route = [
    (-3.7038, 40.4168),
    (-3.6950, 40.4200),
    (-3.6860, 40.4250),
]

zone_ring = [
    (-3.7200, 40.4000),
    (-3.6800, 40.4000),
    (-3.6800, 40.4400),
    (-3.7200, 40.4400),
    (-3.7200, 40.4000),
]

zoo_rows = [
    (
        "madrid_store",
        "Madrid Store",
        "POINT",
        "one location",
        "POINT(-3.7038 40.4168)",
        wkb_point(*madrid),
    ),
    (
        "delivery_route",
        "Tiny Delivery Route",
        "LINESTRING",
        "a connected path",
        "LINESTRING(-3.7038 40.4168, -3.6950 40.4200, -3.6860 40.4250)",
        wkb_linestring(route),
    ),
    (
        "delivery_zone",
        "Tiny Delivery Zone",
        "POLYGON",
        "an enclosed area",
        "POLYGON((-3.72 40.40, -3.68 40.40, -3.68 40.44, -3.72 40.44, -3.72 40.40))",
        wkb_polygon(zone_ring),
    ),
]

zoo = spark.createDataFrame(
    zoo_rows,
    ["object_id", "name", "shape_kind", "real_world_meaning", "wkt_for_humans", "geom_wkb"],
)

print("🌍 THE SPATIAL ZOO")
zoo.select(
    "name",
    "shape_kind",
    "real_world_meaning",
    "wkt_for_humans",
).show(truncate=False)

print("👽 Same objects as machine payloads (WKB hex prefix):")
zoo.select(
    "name",
    "shape_kind",
    F.substring(F.hex("geom_wkb"), 1, 42).alias("wkb_hex_prefix"),
).show(truncate=False)

🌍 THE SPATIAL ZOO
+-------------------+----------+------------------+--------------------------------------------------------------------------+
|name               |shape_kind|real_world_meaning|wkt_for_humans                                                            |
+-------------------+----------+------------------+--------------------------------------------------------------------------+
|Madrid Store       |POINT     |one location      |POINT(-3.7038 40.4168)                                                    |
|Tiny Delivery Route|LINESTRING|a connected path  |LINESTRING(-3.7038 40.4168, -3.6950 40.4200, -3.6860 40.4250)             |
|Tiny Delivery Zone |POLYGON   |an enclosed area  |POLYGON((-3.72 40.40, -3.68 40.40, -3.68 40.44, -3.72 40.44, -3.72 40.40))|
+-------------------+----------+------------------+--------------------------------------------------------------------------+

👽 Same objects as machine payloads (WKB hex prefix):
+-------------------+----------+-------

### What just happened?

Nothing spatial has happened **inside Spark** yet.

That distinction matters.

We created perfectly valid geometry bytes in Python and gave them to a DataFrame.

At this moment Spark can transport the bytes.

But does Spark know that one row is a Point, another a route, and another a Polygon?

Let's ask the schema.

In [3]:
print("📦 RAW DATAFRAME SCHEMA")
zoo.printSchema()

raw_contract = zoo.select(
    F.expr("typeof(geom_wkb)").alias("spark_type"),
).first()["spark_type"]

print(f"\n🧠 What does Spark call geom_wkb?  {raw_contract}")

assert raw_contract.lower() == "binary"

print("""
📡 Mission Control translation:

    Humans:  "That is Madrid, a route, and a delivery zone."
    Spark:   "Excellent. Three byte arrays."
""")

📦 RAW DATAFRAME SCHEMA
root
 |-- object_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- shape_kind: string (nullable = true)
 |-- real_world_meaning: string (nullable = true)
 |-- wkt_for_humans: string (nullable = true)
 |-- geom_wkb: binary (nullable = true)


🧠 What does Spark call geom_wkb?  binary

📡 Mission Control translation:

    Humans:  "That is Madrid, a route, and a delivery zone."
    Spark:   "Excellent. Three byte arrays."



# 2 — Same Shapes, New Contract: `BINARY` → `GEOMETRY(4326)` 🧑‍🚀

Now we use Spark 4.2's native constructor:

```python
st_geomfromwkb(...)
```

It parses WKB and returns a native Spark `GEOMETRY` value.

We also provide:

```text
4326
```

That number is an **SRID — Spatial Reference Identifier**.

For this first lab, keep the definition deliberately small:

> **The SRID identifies the coordinate reference system that gives the coordinate numbers their spatial meaning.**

We are **not** learning projection mathematics yet. The next lab will deliberately break SRIDs, axis order, and reprojection assumptions.

Today we only want to prove two things:

```text
BINARY
→ Spark knows "bytes"

GEOMETRY(4326)
→ Spark knows "spatial value + spatial reference"
```

In [4]:
native = zoo.withColumn(
    "geom",
    F.st_geomfromwkb("geom_wkb", 4326),
)

print("🌍 NATIVE SPATIAL SCHEMA")
native.printSchema()

geom_field = native.schema["geom"]
geom_dtype = geom_field.dataType
geom_dtype_name = type(geom_dtype).__name__
geom_dtype_simple = geom_dtype.simpleString()

print("🧠 PySpark DataType object")
print(f"  ├─ Python class : {geom_dtype_name}")
print(f"  └─ Spark type   : {geom_dtype_simple}")

telemetry = native.select(
    "name",
    "shape_kind",
    F.expr("typeof(geom_wkb)").alias("before_type"),
    F.expr("typeof(geom)").alias("after_type"),
    F.st_srid("geom").alias("srid"),
    (F.hex("geom_wkb") == F.hex(F.st_asbinary("geom"))).alias("wkb_roundtrip_equal"),
).orderBy("name")

print("\n🛰️ TYPE-CONTRACT TELEMETRY")
telemetry.show(truncate=False)

rows = telemetry.collect()

assert len(rows) == 3
assert geom_dtype_name == "GeometryType", (geom_dtype_name, geom_dtype_simple)
assert all(r.before_type.lower() == "binary" for r in rows)
assert all("geometry" in r.after_type.lower() for r in rows)
assert all("4326" in r.after_type for r in rows)
assert all(r.srid == 4326 for r in rows)
assert all(r.wkb_roundtrip_equal for r in rows)

lab_results = {
    "fingerprint": fingerprint,
    "shape_count": len(rows),
    "shape_kinds": sorted(r.shape_kind for r in rows),
    "before_types": sorted(set(r.before_type for r in rows)),
    "after_types": sorted(set(r.after_type for r in rows)),
    "geom_dtype_name": geom_dtype_name,
    "geom_dtype_simple": geom_dtype_simple,
    "srids": sorted(set(r.srid for r in rows)),
    "all_roundtrips_equal": all(r.wkb_roundtrip_equal for r in rows),
}

print("""
✅ Same spatial payload.
✅ Different Spark type contract.
✅ Schema contains a real PySpark GeometryType.
✅ SRID is queryable.
✅ Point, LineString, and Polygon survived the WKB → GEOMETRY → WKB round trip.
""")

🌍 NATIVE SPATIAL SCHEMA
root
 |-- object_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- shape_kind: string (nullable = true)
 |-- real_world_meaning: string (nullable = true)
 |-- wkt_for_humans: string (nullable = true)
 |-- geom_wkb: binary (nullable = true)
 |-- geom: geometry(4326) (nullable = true)

🧠 PySpark DataType object
  ├─ Python class : GeometryType
  └─ Spark type   : geometry(4326)

🛰️ TYPE-CONTRACT TELEMETRY
+-------------------+----------+-----------+--------------+----+-------------------+
|name               |shape_kind|before_type|after_type    |srid|wkb_roundtrip_equal|
+-------------------+----------+-----------+--------------+----+-------------------+
|Madrid Store       |POINT     |binary     |geometry(4326)|4326|true               |
|Tiny Delivery Route|LINESTRING|binary     |geometry(4326)|4326|true               |
|Tiny Delivery Zone |POLYGON   |binary     |geometry(4326)|4326|true               |
+-------------------+----------+-------

# 3 — Let Python Meet a Native Geometry 🐍👨‍🚀

A native Spark datatype should survive beyond SQL syntax.

We collect **one tiny row**—safe, intentional, and not an invitation to `collect()` the European continent.

If PySpark understands the native spatial type, the `geom` column should arrive as a Python `Geometry` object.

If we explicitly call:

```python
st_asbinary(...)
```

we should get ordinary Python `bytes`.

Same shape.

Different contract.

In [5]:
madrid_row = (
    native
    .filter(F.col("object_id") == "madrid_store")
    .select(
        "geom",
        F.st_asbinary("geom").alias("as_binary"),
    )
    .first()
)

native_python_type = type(madrid_row.geom).__name__
binary_python_type = type(madrid_row.as_binary).__name__
python_srid = madrid_row.geom.getSrid()
same_python_payload = madrid_row.geom.getBytes() == madrid_row.as_binary

print("🐍 Python border checkpoint")
print(f"  ├─ native column object : {native_python_type}")
print(f"  ├─ ST_AsBinary object   : {binary_python_type}")
print(f"  ├─ native SRID          : {python_srid}")
print(f"  └─ same WKB payload     : {same_python_payload}")

assert native_python_type == "Geometry"
assert binary_python_type == "bytes"
assert python_srid == 4326
assert same_python_payload

lab_results.update({
    "native_python_type": native_python_type,
    "binary_python_type": binary_python_type,
    "python_srid": python_srid,
    "same_python_payload": same_python_payload,
})

🐍 Python border checkpoint
  ├─ native column object : Geometry
  ├─ ST_AsBinary object   : bytes
  ├─ native SRID          : 4326
  └─ same WKB payload     : True


# 4 — Ask the First Real Spatial Question 🏪📍🗺️

We now have:

```text
Madrid Store        → POINT
Delivery Zone       → POLYGON
```

The obvious spatial question is:

> **Is the Madrid Store inside the Delivery Zone?**

In spatial SQL, that relationship is commonly expressed with:

```sql
ST_Within(point, polygon)
```

Spark 4.2 now understands both columns as native Geometry.

But does **stock Spark 4.2** also provide the spatial predicate?

For a published notebook we do **not** need to deliberately execute a missing routine and dump a giant JVM stack trace just to prove absence.

Instead we ask Spark's own function catalog in two independent ways:

```text
Catalog.listFunctions()
SHOW FUNCTIONS LIKE 'st_within'
```

If both say the function is absent, that is cleaner evidence than manufacturing an exception.

In [6]:
native.createOrReplaceTempView("spatial_zoo")

# Probe 1: PySpark's function catalog
catalog_matches = sorted(
    f.name
    for f in spark.catalog.listFunctions()
    if f.name.lower() == "st_within"
)

# Probe 2: Spark SQL's SHOW FUNCTIONS
show_function_rows = spark.sql("SHOW FUNCTIONS LIKE 'st_within'").collect()
show_matches = sorted(row[0] for row in show_function_rows)

st_within_available = bool(catalog_matches or show_matches)

print("🔭 Capability probe: ST_Within")
print(f"  ├─ Catalog.listFunctions matches : {catalog_matches}")
print(f"  ├─ SHOW FUNCTIONS matches        : {show_matches}")
print(f"  └─ Registered in stock Spark 4.2 : {st_within_available}")

assert catalog_matches == [], catalog_matches
assert show_matches == [], show_matches
assert st_within_available is False

lab_results.update({
    "st_within_available": st_within_available,
    "st_within_catalog_matches": catalog_matches,
    "st_within_show_matches": show_matches,
})

print("""
🧠 Translation:

Spark now understands the NOUN:
    GEOMETRY

But stock Spark 4.2 does not register this VERB:
    ST_Within

No fake failure required. Spark's own catalogs give us the answer.
""")

🔭 Capability probe: ST_Within
  ├─ Catalog.listFunctions matches : []
  ├─ SHOW FUNCTIONS matches        : []
  └─ Registered in stock Spark 4.2 : False

🧠 Translation:

Spark now understands the NOUN:
    GEOMETRY

But stock Spark 4.2 does not register this VERB:
    ST_Within

No fake failure required. Spark's own catalogs give us the answer.



## 📻 So where does Apache Sedona fit?

This notebook intentionally contains **no Sedona runtime**.

That is important: we want a clean boundary test of **stock Spark 4.2**.

Historically, Apache Sedona supplied the much broader geospatial layer around Spark: predicates such as `ST_Within` and `ST_Intersects`, measurements such as distance, CRS transformation, spatial joins, indexes, and much more.

So the beginner-friendly picture is:

```text
              STOCK SPARK 4.2
        ┌──────────────────────────┐
        │ GEOMETRY / GEOGRAPHY     │
        │ SRID-aware types         │
        │ WKB parsing              │
        │ spatial storage plumbing │
        └─────────────┬────────────┘
                      │
                      │ richer spatial computation
                      ▼
        ┌──────────────────────────┐
        │       Apache Sedona      │
        │ predicates / distance    │
        │ CRS transformation       │
        │ spatial joins / indexes  │
        └──────────────────────────┘
```

That is a **conceptual layering diagram**, not an internal class diagram.

The first-contact lesson is:

> **Spark 4.2 has learned what spatial data *is*. A specialized spatial engine still knows many more things to *do* with it.**

We will investigate that boundary properly later.

# 📊 Post-Lab Analysis — Build the Verdict From What Actually Happened

The next cell renders the conclusion using the values observed during **this execution**.

If one of the assertions above fails, the notebook stops instead of printing a reassuring story about another universe.

In [7]:
from IPython.display import Markdown, display

shapes = ", ".join(f"`{x}`" for x in lab_results["shape_kinds"])
before = ", ".join(f"`{x}`" for x in lab_results["before_types"])
after = ", ".join(f"`{x}`" for x in lab_results["after_types"])
srids = ", ".join(str(x) for x in lab_results["srids"])

analysis = f"""
# 📊 Post-Lab Analysis: Spark Has Learned the Difference Between “Bytes” and “Place”

We began with **{lab_results['shape_count']} spatial objects**: {shapes}.

To us they represented a store location, a route, and a delivery zone.  
Before native parsing, Spark described every machine payload simply as **{before}**.

After `st_geomfromwkb(..., 4326)`, Spark described the same payload as **{after}**. The schema exposed a real **`{lab_results['geom_dtype_name']}`**, and `st_srid` reported **SRID {srids}**.

### 1. A Shape Can Exist Before Spark Understands the Shape

The WKB was already valid spatial data outside Spark. Spark could store and move it perfectly well as `BINARY`.

But `BINARY` only told Spark:

> “Here are some bytes.”

After parsing into native Geometry, the schema itself told Spark:

> “This is a spatial value, and its coordinate reference is identified by SRID 4326.”

Nothing about Madrid changed.

**Spark's understanding of Madrid changed.**

### 2. Native Spatial Is More Than a Fancy Column Name

All three shapes completed the WKB → `GEOMETRY(4326)` → WKB round trip unchanged:

**{lab_results['all_roundtrips_equal']}**

The Python boundary preserved the distinction too. Collecting the native value produced a **`{lab_results['native_python_type']}`** carrying SRID **{lab_results['python_srid']}**, while `st_asbinary` produced ordinary **`{lab_results['binary_python_type']}`**.

Same shape.  
Same WKB.  
Different contract.

### 3. Spark 4.2 Knows the Noun — Not Yet Every Verb

We asked Spark's own catalogs whether `ST_Within` was registered.

```text
Catalog.listFunctions() : {lab_results['st_within_catalog_matches']}
SHOW FUNCTIONS           : {lab_results['st_within_show_matches']}
```

Both probes agreed:

**`ST_Within` available: `{lab_results['st_within_available']}`**

That is not a limitation in the experiment. It is one of the most important results.

Stock Spark 4.2 can natively represent the Point and Polygon, preserve their SRID, carry them through PySpark, and round-trip their WKB.

But it still does not register the `ST_Within` predicate needed to answer:

> “Is the store inside the delivery zone?”

This is the boundary between:

```text
understanding spatial DATA
```

and:

```text
providing a complete spatial ENGINE
```

> ## 🚀 Mission Verdict
> **Spark 4.2 has learned what a spatial value is.** A `POINT`, `LINESTRING`, or `POLYGON` no longer has to masquerade as anonymous `BINARY`. But knowing that something is Geometry is not the same as knowing every GIS operation that can be performed on it.
>
> **Space has entered the Spark type system. The rest of the galaxy is still ahead.**
"""

display(Markdown(analysis))


# 📊 Post-Lab Analysis: Spark Has Learned the Difference Between “Bytes” and “Place”

We began with **3 spatial objects**: `LINESTRING`, `POINT`, `POLYGON`.

To us they represented a store location, a route, and a delivery zone.  
Before native parsing, Spark described every machine payload simply as **`binary`**.

After `st_geomfromwkb(..., 4326)`, Spark described the same payload as **`geometry(4326)`**. The schema exposed a real **`GeometryType`**, and `st_srid` reported **SRID 4326**.

### 1. A Shape Can Exist Before Spark Understands the Shape

The WKB was already valid spatial data outside Spark. Spark could store and move it perfectly well as `BINARY`.

But `BINARY` only told Spark:

> “Here are some bytes.”

After parsing into native Geometry, the schema itself told Spark:

> “This is a spatial value, and its coordinate reference is identified by SRID 4326.”

Nothing about Madrid changed.

**Spark's understanding of Madrid changed.**

### 2. Native Spatial Is More Than a Fancy Column Name

All three shapes completed the WKB → `GEOMETRY(4326)` → WKB round trip unchanged:

**True**

The Python boundary preserved the distinction too. Collecting the native value produced a **`Geometry`** carrying SRID **4326**, while `st_asbinary` produced ordinary **`bytes`**.

Same shape.  
Same WKB.  
Different contract.

### 3. Spark 4.2 Knows the Noun — Not Yet Every Verb

We asked Spark's own catalogs whether `ST_Within` was registered.

```text
Catalog.listFunctions() : []
SHOW FUNCTIONS           : []
```

Both probes agreed:

**`ST_Within` available: `False`**

That is not a limitation in the experiment. It is one of the most important results.

Stock Spark 4.2 can natively represent the Point and Polygon, preserve their SRID, carry them through PySpark, and round-trip their WKB.

But it still does not register the `ST_Within` predicate needed to answer:

> “Is the store inside the delivery zone?”

This is the boundary between:

```text
understanding spatial DATA
```

and:

```text
providing a complete spatial ENGINE
```

> ## 🚀 Mission Verdict
> **Spark 4.2 has learned what a spatial value is.** A `POINT`, `LINESTRING`, or `POLYGON` no longer has to masquerade as anonymous `BINARY`. But knowing that something is Geometry is not the same as knowing every GIS operation that can be performed on it.
>
> **Space has entered the Spark type system. The rest of the galaxy is still ahead.**


# 🛰️ Mission Handoff

We deliberately did **not** explain SRID 4326 deeply here.

We only proved that it exists in the type contract and survives into Python.

That is the next mission:

```text
GEOMETRY(4326)
GEOMETRY(3857)
GEOMETRY(0)
GEOMETRY(ANY)
GEOGRAPHY(4326)
```

Why can the same place have different coordinates?

Why can valid coordinates still be wrong?

Why is `ST_SetSrid` absolutely not reprojection?

And, most importantly:

```text
where_the_hell_is_madrid()
```

In [8]:
# Clean shutdown. The evidence is already in the notebook.
spark.stop()
print("🌍 Spark stopped. First-contact mission complete.")

🌍 Spark stopped. First-contact mission complete.
